![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## RollingWindow Warm-Up Research

Warm a daily TradeBar window from history before building close-direction series.

### Set Up QuantBook

Create a daily SPY subscription for the history request.

In [ ]:
qb = QuantBook()
qb.set_start_date(2024, 9, 1)
qb.settings.seed_initial_prices = True
equity = qb.add_equity("SPY", Resolution.DAILY)

### Warm Rolling Window

Fill the 20-bar window with typed daily history before using it.

In [ ]:
window = RollingWindow[TradeBar](20)
# Warm the rolling window with typed daily history.
history = qb.history[TradeBar](equity, 20, Resolution.DAILY)
for bar in history:
    window.add(bar)

### Verify Readiness

Confirm the window has all 20 bars before comparing the latest two closes.

In [ ]:
print(f"Window count: {window.count}")
print(f"Window ready: {window.is_ready}")
assert window.count == 20
assert window.is_ready

### Build Time Series

Replay a longer history window and store the latest and previous close values.

In [ ]:
series_window = RollingWindow[TradeBar](20)
value_rows = []
signal_rows = []
for bar in qb.history[TradeBar](equity, 120, Resolution.DAILY):
    series_window.add(bar)
    if not series_window.is_ready:
        continue
    latest_close = series_window[0].close
    previous_close = series_window[1].close
    value_rows.append({"time": bar.end_time, "latest_close": latest_close, "previous_close": previous_close})
    signal_rows.append({"time": bar.end_time, "signal": 1 if latest_close > previous_close else 0})

indicator_values = pd.DataFrame(value_rows).set_index("time")
indicator_values.tail()

### Signal Series

Display the 1/0 signal generated from the latest-close-above-previous-close rule.

In [ ]:
signals = pd.DataFrame(signal_rows).set_index("time")
signals.tail()